# SmartCare Option C - Preprocessing, Feature Engineering and Model Development

This notebook continues the verified dataset-understanding stage. It compares clinical-only and context-augmented feature sets while keeping the final 20% test set untouched until model selection and tuning are complete.

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay, RocCurveDisplay, accuracy_score,
    balanced_accuracy_score, classification_report, cohen_kappa_score,
    confusion_matrix, f1_score, mean_absolute_error, precision_score,
    recall_score, roc_auc_score, roc_curve
)
from sklearn.model_selection import (
    RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42
CLASS_NAMES = ["Low", "Medium", "High"]
CLASS_TO_INT = {name: index for index, name in enumerate(CLASS_NAMES)}
INT_TO_CLASS = {index: name for name, index in CLASS_TO_INT.items()}

## 1. Load data and establish the prediction target

In [ ]:
PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "smartcare_ai_dataset_1000.csv"
MODEL_DIR = PROJECT_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA_PATH)
TARGET = "disease_risk_level"

assert set(df[TARGET].unique()) == set(CLASS_NAMES)
assert df[TARGET].isna().sum() == 0

y = df[TARGET].map(CLASS_TO_INT).astype(int)
print(f"Loaded {len(df):,} records and {df.shape[1]} columns")
display(pd.DataFrame({"count": df[TARGET].value_counts().reindex(CLASS_NAMES),
                      "percentage": (df[TARGET].value_counts(normalize=True).reindex(CLASS_NAMES) * 100).round(1)}))

## 2. Leakage-safe feature sets

The primary model uses information plausibly available at initial assessment. The context experiment adds diagnosis, department and patient-history variables. Post-admission, treatment, billing, payment and other target variables remain excluded.

In [ ]:
def add_engineered_features(data):
    """Create deterministic, clinically interpretable features without using the target."""
    result = data.copy()
    result["pulse_pressure"] = result["systolic_bp"] - result["diastolic_bp"]
    result["mean_arterial_pressure"] = (
        result["systolic_bp"] + 2 * result["diastolic_bp"]
    ) / 3
    previous = result["previous_appointments"].replace(0, np.nan)
    result["missed_appointment_ratio"] = (
        result["missed_previous_appointments"] / previous
    ).fillna(0).clip(0, 1)
    result["has_previous_admission"] = (result["previous_admissions"] > 0).astype(int)
    return result

model_df = add_engineered_features(df)

clinical_numeric = [
    "age", "previous_admissions", "systolic_bp", "diastolic_bp",
    "blood_sugar_mg_dl", "cholesterol_mg_dl", "bmi",
    "pulse_pressure", "mean_arterial_pressure", "has_previous_admission"
]
clinical_categorical = ["gender"]

context_numeric = clinical_numeric + [
    "previous_appointments", "missed_previous_appointments",
    "missed_appointment_ratio"
]
context_categorical = clinical_categorical + ["blood_group", "department", "diagnosis"]

feature_sets = {
    "Clinical only": {"numeric": clinical_numeric, "categorical": clinical_categorical},
    "Context augmented": {"numeric": context_numeric, "categorical": context_categorical},
}

for name, groups in feature_sets.items():
    columns = groups["numeric"] + groups["categorical"]
    print(f"{name}: {len(columns)} input features")
    print(columns)

## 3. Create one untouched stratified test set

In [ ]:
train_index, test_index = train_test_split(
    np.arange(len(model_df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_all = model_df.iloc[train_index].reset_index(drop=True)
X_test_all = model_df.iloc[test_index].reset_index(drop=True)
y_train = y.iloc[train_index].reset_index(drop=True)
y_test = y.iloc[test_index].reset_index(drop=True)

split_check = pd.DataFrame({
    "Full dataset": y.value_counts(normalize=True).sort_index(),
    "Training set": y_train.value_counts(normalize=True).sort_index(),
    "Test set": y_test.value_counts(normalize=True).sort_index(),
}, index=range(3))
split_check.index = CLASS_NAMES
display((split_check * 100).round(1).rename_axis("Risk class (%)"))
print(f"Training rows: {len(y_train)} | Untouched test rows: {len(y_test)}")

## 4. Reusable preprocessing pipelines

Numerical values are median-imputed and standardized. Categorical values are mode-imputed and one-hot encoded. All preprocessing is fitted inside each cross-validation fold.

In [ ]:
def build_preprocessor(numeric_features, categorical_features):
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features),
    ], remainder="drop")

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=5, class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=350, min_samples_leaf=2, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        objective="multi:softprob", num_class=3, eval_metric="mlogloss",
        n_estimators=250, max_depth=3, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85, reg_lambda=1.0,
        random_state=RANDOM_STATE, n_jobs=1
    ),
}

pipelines = {}
for feature_set_name, groups in feature_sets.items():
    preprocessor = build_preprocessor(groups["numeric"], groups["categorical"])
    for model_name, estimator in models.items():
        pipelines[(feature_set_name, model_name)] = Pipeline([
            ("preprocessor", clone(preprocessor)),
            ("model", clone(estimator)),
        ])
print(f"Prepared {len(pipelines)} leakage-safe model pipelines")

## 5. Compare four algorithms across both feature sets

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
    "precision_macro": "precision_macro",
    "recall_macro": "recall_macro",
    "f1_macro": "f1_macro",
    "f1_weighted": "f1_weighted",
    "roc_auc_ovr": "roc_auc_ovr",
}

comparison_rows = []
for (feature_set_name, model_name), pipeline in pipelines.items():
    selected_features = (feature_sets[feature_set_name]["numeric"]
                         + feature_sets[feature_set_name]["categorical"])
    cv_result = cross_validate(
        pipeline, X_train_all[selected_features], y_train,
        cv=cv, scoring=scoring, n_jobs=1, return_train_score=False,
    )
    row = {"Feature set": feature_set_name, "Model": model_name}
    for metric in scoring:
        values = cv_result[f"test_{metric}"]
        row[f"{metric} mean"] = values.mean()
        row[f"{metric} std"] = values.std(ddof=1)
    comparison_rows.append(row)
    print(f"Finished: {feature_set_name} | {model_name} | macro-F1={row['f1_macro mean']:.3f}")

comparison = pd.DataFrame(comparison_rows).sort_values("f1_macro mean", ascending=False)
display(comparison.round(3))

In [ ]:
plt.figure(figsize=(11, 5))
sns.barplot(data=comparison, x="Model", y="f1_macro mean", hue="Feature set")
plt.ylim(0, 1)
plt.ylabel("Mean 5-fold macro-F1")
plt.title("Cross-validated Model Comparison")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

# 6. Tune the strongest training-stage candidate

The winning candidate is selected only from cross-validation on the training partition. Its hyperparameters are then tuned with another stratified training-only search.

In [ ]:
winner_row = comparison.iloc[0]
winner_feature_set = winner_row["Feature set"]
winner_model_name = winner_row["Model"]
winner_features = (feature_sets[winner_feature_set]["numeric"]
                   + feature_sets[winner_feature_set]["categorical"])
winner_pipeline = clone(pipelines[(winner_feature_set, winner_model_name)])

parameter_spaces = {
    "Logistic Regression": {
        "model__C": np.logspace(-2, 2, 20),
    },
    "Decision Tree": {
        "model__criterion": ["gini", "entropy"],
        "model__max_depth": [3, 4, 5, 6, 8, 10, None],
        "model__min_samples_leaf": [1, 2, 3, 5, 8, 12],
        "model__min_samples_split": [2, 5, 10, 20],
    },
    "Random Forest": {
        "model__n_estimators": [200, 300, 450, 600],
        "model__max_depth": [None, 6, 8, 10, 14],
        "model__min_samples_leaf": [1, 2, 3, 5],
        "model__max_features": ["sqrt", "log2", 0.7],
    },
    "XGBoost": {
        "model__n_estimators": [150, 250, 350, 500],
        "model__max_depth": [2, 3, 4, 5],
        "model__learning_rate": [0.02, 0.04, 0.06, 0.1],
        "model__subsample": [0.75, 0.85, 1.0],
        "model__colsample_bytree": [0.75, 0.85, 1.0],
        "model__reg_lambda": [0.5, 1.0, 2.0, 5.0],
    },
}

search = RandomizedSearchCV(
    estimator=winner_pipeline,
    param_distributions=parameter_spaces[winner_model_name],
    n_iter=16, scoring="f1_macro", cv=cv,
    random_state=RANDOM_STATE, n_jobs=1, refit=True, verbose=0,
)
search.fit(X_train_all[winner_features], y_train)
best_pipeline = search.best_estimator_

print(f"Selected feature set: {winner_feature_set}")
print(f"Selected algorithm: {winner_model_name}")
print(f"Best training CV macro-F1: {search.best_score_:.3f}")
print("Best hyperparameters:")
display(pd.Series(search.best_params_, name="value").to_frame())

# 7. Final evaluation on the untouched test set

This is the first point at which the final test labels are used for performance assessment.


In [ ]:
X_test = X_test_all[winner_features]
test_predictions = best_pipeline.predict(X_test)
test_probabilities = best_pipeline.predict_proba(X_test)

test_metrics = pd.Series({
    "Accuracy": accuracy_score(y_test, test_predictions),
    "Balanced accuracy": balanced_accuracy_score(y_test, test_predictions),
    "Macro precision": precision_score(y_test, test_predictions, average="macro", zero_division=0),
    "Macro recall": recall_score(y_test, test_predictions, average="macro", zero_division=0),
    "Macro F1": f1_score(y_test, test_predictions, average="macro", zero_division=0),
    "Weighted F1": f1_score(y_test, test_predictions, average="weighted", zero_division=0),
    "Macro ROC-AUC (OvR)": roc_auc_score(y_test, test_probabilities, multi_class="ovr", average="macro"),
    "Quadratic weighted kappa": cohen_kappa_score(y_test, test_predictions, weights="quadratic"),
    "Ordinal MAE": mean_absolute_error(y_test, test_predictions),
    "Severe Low-High error rate": np.mean(np.abs(y_test.to_numpy() - test_predictions) == 2),
})
display(test_metrics.round(3).to_frame("Test value"))

report = pd.DataFrame(classification_report(
    y_test, test_predictions, labels=[0, 1, 2], target_names=CLASS_NAMES,
    output_dict=True, zero_division=0,
)).T
display(report.round(3))